# 📊 EDA - Saber Pro 2018–2024
**Análisis exploratorio:** USTA vs Nacional  
Genera gráficas y reporte CSV en `notebooks/eda_saber_pro/`

## 0. Configuración de rutas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
from pathlib import Path

matplotlib.use('Agg')  # evita errores si no hay pantalla

# ─────────────────────────────────────────────
BASE        = Path().resolve()                          # carpeta raíz del repo
RUTA_CSV    = BASE / 'output' / 'consolidado_2018_2024.csv'
CARPETA_OUT = BASE / 'notebooks' / 'eda_saber_pro'

CARPETA_OUT.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────

print(f'CSV:        {RUTA_CSV}')
print(f'Salida EDA: {CARPETA_OUT}')

## 1. Carga de datos

In [ ]:
print('Cargando datos...')
df = pd.read_csv(RUTA_CSV, low_memory=False)
print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')
df.head(3)

## 2. Resumen general

In [ ]:
print('--- RESUMEN GENERAL ---')
print(f"Periodos disponibles: {sorted(df['periodo'].unique())}")
print(f"Registros USTA:       {df['es_usta'].sum():,}")
print(f"Registros nacional:   {(df['es_usta']==0).sum():,}")
print(f"\nValores nulos por columna (solo columnas con nulos):")
nulos = df.isnull().sum()
print(nulos[nulos > 0])

## 3. Puntajes promedio: USTA vs Nacional

In [ ]:
MODULOS = {
    'punt_global':              'Puntaje Global',
    'mod_lectura_critica_punt': 'Lectura Crítica',
    'mod_razona_cuantitat_punt':'Razonamiento Cuantitativo',
    'mod_comuni_escrita_punt':  'Comunicación Escrita',
    'mod_ingles_punt':          'Inglés',
    'mod_competen_ciudada_punt':'Competencias Ciudadanas',
}

resumen = []
for col, nombre in MODULOS.items():
    if col in df.columns:
        media_usta = df[df['es_usta']==1][col].mean()
        media_nac  = df[df['es_usta']==0][col].mean()
        resumen.append({
            'modulo': nombre,
            'promedio_usta': round(media_usta, 2),
            'promedio_nacional': round(media_nac, 2),
            'diferencia': round(media_usta - media_nac, 2)
        })

df_resumen = pd.DataFrame(resumen)
print('--- COMPARACIÓN USTA VS NACIONAL ---')
display(df_resumen)

df_resumen.to_csv(CARPETA_OUT / 'comparacion_usta_nacional.csv', index=False)
print('Guardado: comparacion_usta_nacional.csv')

## 4. Evolución temporal: USTA vs Nacional

In [ ]:
evolucion = df.groupby(['periodo', 'es_usta'])['punt_global'].mean().reset_index()
evolucion['grupo'] = evolucion['es_usta'].map({1: 'USTA', 0: 'Nacional'})

fig, ax = plt.subplots(figsize=(12, 5))
for grupo, datos in evolucion.groupby('grupo'):
    ax.plot(datos['periodo'], datos['punt_global'], marker='o', label=grupo, linewidth=2)
ax.set_title('Evolución del Puntaje Global: USTA vs Nacional')
ax.set_xlabel('Periodo')
ax.set_ylabel('Puntaje Promedio')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CARPETA_OUT / 'evolucion_temporal.png', dpi=150)
plt.show()
print('Guardada: evolucion_temporal.png')

## 5. Comparación por módulo (barras)

In [ ]:
modulos_disp = [m for m in MODULOS.keys() if m in df.columns and m != 'punt_global']
df_bar = df.groupby('es_usta')[modulos_disp].mean().T
df_bar.columns = ['Nacional', 'USTA']
df_bar.index = [MODULOS[m] for m in df_bar.index]

ax = df_bar.plot(kind='bar', figsize=(12, 6), color=['#4C72B0', '#DD8452'])
ax.set_title('Puntaje Promedio por Módulo: USTA vs Nacional')
ax.set_ylabel('Puntaje Promedio')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(CARPETA_OUT / 'comparacion_modulos.png', dpi=150)
plt.show()
print('Guardada: comparacion_modulos.png')

## 6. Distribución del puntaje global

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df[df['es_usta']==0]['punt_global'].dropna().plot(
    kind='hist', bins=50, alpha=0.6, label='Nacional', color='#4C72B0', ax=ax)
df[df['es_usta']==1]['punt_global'].dropna().plot(
    kind='hist', bins=50, alpha=0.6, label='USTA', color='#DD8452', ax=ax)
ax.set_title('Distribución del Puntaje Global')
ax.set_xlabel('Puntaje Global')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.savefig(CARPETA_OUT / 'distribucion_puntaje.png', dpi=150)
plt.show()
print('Guardada: distribucion_puntaje.png')

## 7. Top programas USTA por puntaje global

In [ ]:
if 'estu_prgm_academico' in df.columns:
    top_programas = (
        df[df['es_usta']==1]
        .groupby('estu_prgm_academico')['punt_global']
        .agg(['mean','count'])
        .reset_index()
        .rename(columns={'mean':'promedio','count':'estudiantes'})
        .query('estudiantes >= 50')
        .sort_values('promedio', ascending=False)
        .head(20)
    )
    top_programas.to_csv(CARPETA_OUT / 'top_programas_usta.csv', index=False)
    print('Top programas USTA guardado: top_programas_usta.csv')
    display(top_programas)
else:
    print('Columna estu_prgm_academico no disponible')

## 8. Resumen final

In [ ]:
archivos_generados = list(CARPETA_OUT.glob('*'))
print(f'EDA completado. Archivos generados en: {CARPETA_OUT}')
for f in sorted(archivos_generados):
    print(f'  - {f.name}')